# 4. Use LlamaIndex's prebuilt ReAct agent

The previous notebook makes retrieval, grading and rewriting explicit. Here LlamaIndex's `ReActAgent` manages the loop and decides when to search again. It uses the same GoodMem collections, reranker, prompts and native retrieval tools.

The code uses the current workflow-based API: construct the agent and `await agent.run(...)`. The older `ReActAgent.from_tools(...).chat(...)` example in the published GoodMem plugin no longer works with the tested LlamaIndex version.

In [ ]:
from goodmem_rag.config import Settings, chat_model
from goodmem_rag.retrieval import make_tools

settings = Settings.from_env()
state = settings.state()
model = chat_model()

In [ ]:
from llama_index.core.agent.workflow import ReActAgent
from goodmem_rag.agents import SYSTEM_PROMPT
from goodmem_rag.evaluation import SEQUENTIAL_QUESTION

async with settings.async_client() as client:
    tools = make_tools(async_client=client, state=state)
    agent = ReActAgent(llm=model, tools=tools, system_prompt=SYSTEM_PROMPT,
                       streaming=False, timeout=180, early_stopping_method="generate")
    response = await agent.run(user_msg=SEQUENTIAL_QUESTION, max_iterations=5)

print(response.response.content)
for call in response.tool_calls:
    print(call.tool_name, call.tool_kwargs)
    print("Sources:", [node.metadata.get("source") for node in call.tool_output.raw_output])

## Compare the two patterns

The explicit workflow gives you a place to enforce a relevance policy or add a review step. ReAct has less application code and lets the model decide how to proceed.

Both use server-side reranking without a GoodMem LLM. The chat model writes the final answer. Run `uv run llamaindex-rag evaluate` to check retrieval, direct answers, single-collection questions, comparisons and dependent searches for both agents. Those checks establish that the port works; comparing answer quality needs a larger controlled evaluation.